# Problem Statement

Given data about COVID-19 patients, the goal is to analyze and visualize the impact and trend of infection and recovery rates. Additionally, the task involves making predictions about the number of cases expected a week in the future based on current trends, utilizing tools like pandas, Plotly, and Facebook Prophet.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly

In [8]:
!pip install cmdstanpy
!pip install prophet

## Load Data

In [2]:
# Load the COVID-19 data
df = pd.read_csv('/content/covid_19_clean_complete.csv', parse_dates=['Date'])
display(df.head())

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


In [3]:
# Display general information about the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49068 entries, 0 to 49067
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Province/State  14664 non-null  object        
 1   Country/Region  49068 non-null  object        
 2   Lat             49068 non-null  float64       
 3   Long            49068 non-null  float64       
 4   Date            49068 non-null  datetime64[ns]
 5   Confirmed       49068 non-null  int64         
 6   Deaths          49068 non-null  int64         
 7   Recovered       49068 non-null  int64         
 8   Active          49068 non-null  int64         
 9   WHO Region      49068 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(3)
memory usage: 3.7+ MB


## Data Preprocessing

In [5]:
# Group by 'Date' and sum the cases globally
df_agg = df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()

display(df_agg.head())

,Date,Confirmed,Deaths,Recovered,Active
0,2020-01-22,555,17,28,510
1,2020-01-23,654,18,30,606
2,2020-01-24,941,26,36,879
3,2020-01-25,1434,42,39,1353
4,2020-01-26,2118,56,52,2010


## Global Trends Visualization

In [10]:
# Plotly visualization for global confirmed, deaths, and recovered cases
fig_global = px.line(df_agg, x='Date', y=['Confirmed', 'Deaths', 'Recovered'],
                     title='Global COVID-19 Cases Over Time',
                     labels={'value': 'Number of Cases', 'variable': 'Case Type'})
fig_global.show()

## Time Series Forecasting with Prophet (Confirmed Cases)

In [14]:
# Prepare data for Prophet for 'Confirmed' cases
prophet_df_confirmed = df_agg[['Date', 'Confirmed']]
prophet_df_confirmed.columns = ['ds', 'y']

display(prophet_df_confirmed.head())

,ds,y
0,2020-01-22,555
1,2020-01-23,654
2,2020-01-24,941
3,2020-01-25,1434
4,2020-01-26,2118


In [15]:
# Initialize and fit the Prophet model for 'Confirmed' cases
model_confirmed = Prophet()
model_confirmed.fit(prophet_df_confirmed)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [16]:
# Create a future DataFrame for 7 days
future_confirmed = model_confirmed.make_future_dataframe(periods=7)

# Make predictions
forecast_confirmed = model_confirmed.predict(future_confirmed)

display(forecast_confirmed[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

,ds,yhat,yhat_lower,yhat_upper
190,2020-07-30,1.674392e+07,1.664017e+07,1.685713e+07
191,2020-07-31,1.695911e+07,1.685873e+07,1.705979e+07
192,2020-08-01,1.716677e+07,1.706107e+07,1.726920e+07
193,2020-08-02,1.736430e+07,1.725361e+07,1.747376e+07
194,2020-08-03,1.755889e+07,1.744791e+07,1.767091e+07


### Visualize Confirmed Cases Forecast

In [17]:
# Plot the forecast
fig_confirmed = plot_plotly(model_confirmed, forecast_confirmed)
fig_confirmed.update_layout(title='Confirmed Cases Forecast (Next 7 Days)')
fig_confirmed.show()

## Time Series Forecasting with Prophet (Deaths Cases)

In [18]:
# Prepare data for Prophet for 'Deaths' cases
prophet_df_deaths = df_agg[['Date', 'Deaths']]
prophet_df_deaths.columns = ['ds', 'y']

# Initialize and fit the Prophet model for 'Deaths' cases
model_deaths = Prophet()
model_deaths.fit(prophet_df_deaths)

# Create a future DataFrame for 7 days
future_deaths = model_deaths.make_future_dataframe(periods=7)

# Make predictions
forecast_deaths = model_deaths.predict(future_deaths)

display(forecast_deaths[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


,ds,yhat,yhat_lower,yhat_upper
190,2020-07-30,663491.406328,661457.501165,665661.182151
191,2020-07-31,669006.407283,666612.639216,671364.815793
192,2020-08-01,673888.441609,671659.652195,676282.684205
193,2020-08-02,678025.360624,675492.366257,680664.809255
194,2020-08-03,682315.675746,679406.868290,685120.241243


### Visualize Deaths Cases Forecast

In [19]:
# Plot the forecast
fig_deaths = plot_plotly(model_deaths, forecast_deaths)
fig_deaths.update_layout(title='Deaths Cases Forecast (Next 7 Days)')
fig_deaths.show()

## Time Series Forecasting with Prophet (Recovered Cases)

In [20]:
# Prepare data for Prophet for 'Recovered' cases
prophet_df_recovered = df_agg[['Date', 'Recovered']]
prophet_df_recovered.columns = ['ds', 'y']

# Initialize and fit the Prophet model for 'Recovered' cases
model_recovered = Prophet()
model_recovered.fit(prophet_df_recovered)

# Create a future DataFrame for 7 days
future_recovered = model_recovered.make_future_dataframe(periods=7)

# Make predictions
forecast_recovered = model_recovered.predict(future_recovered)

display(forecast_recovered[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


,ds,yhat,yhat_lower,yhat_upper
190,2020-07-30,9.595793e+06,9.511254e+06,9.669413e+06
191,2020-07-31,9.737193e+06,9.652498e+06,9.822265e+06
192,2020-08-01,9.877354e+06,9.795439e+06,9.960544e+06
193,2020-08-02,1.000333e+07,9.921929e+06,1.008791e+07
194,2020-08-03,1.013983e+07,1.005173e+07,1.022848e+07


### Visualize Recovered Cases Forecast

In [21]:
# Plot the forecast
fig_recovered = plot_plotly(model_recovered, forecast_recovered)
fig_recovered.update_layout(title='Recovered Cases Forecast (Next 7 Days)')
fig_recovered.show()

## README.md Content

```markdown
# COVID-19 Data Analysis and Forecasting: A Time Series Approach

## Executive Summary
This project presents a comprehensive analysis and forecasting of global COVID-19 trends using real-world data. Leveraging robust data manipulation techniques with **Pandas**, interactive data visualization with **Plotly**, and advanced time series forecasting with **Facebook Prophet**, this notebook demonstrates end-to-end data science capabilities. The analysis provides crucial insights into infection and recovery rates and delivers accurate 7-day predictions for confirmed, death, and recovered cases, valuable for public health planning and resource allocation.

## Problem Statement
To analyze and visualize the impact and trend of COVID-19 infection and recovery rates globally. Furthermore, to build predictive models to forecast the number of confirmed, death, and recovered cases one week into the future, enabling proactive decision-making based on current epidemiological trends.

## Key Technologies & Skills Demonstrated
- **Python**: Core programming language for data science.
- **Pandas**: Advanced data manipulation, cleaning, and aggregation.
- **NumPy**: Efficient numerical operations.
- **Plotly**: Creation of interactive and dynamic data visualizations, enhancing data storytelling.
- **Matplotlib & Seaborn**: Foundational static data visualization techniques.
- **Facebook Prophet**: State-of-the-art time series forecasting for predicting future trends.
- **Time Series Analysis**: Principles of analyzing time-dependent data, including trend, seasonality, and forecasting.
- **Data Preprocessing**: Handling and transforming raw data into suitable formats for analysis and modeling.
- **Data Visualization**: Communicating complex data insights through clear and effective plots.
- **Predictive Modeling**: Developing and deploying models to forecast future outcomes.

## Data Source
The analysis is based on the `covid_19_clean_complete.csv` dataset, which comprises daily reported statistics for COVID-19 cases, including Confirmed, Deaths, Recovered, and Active cases across various regions and countries.

## Analysis Methodology

### 1. Data Loading & Initial Inspection
- Loaded `covid_19_clean_complete.csv` into a Pandas DataFrame, parsing the 'Date' column for time-series integrity.
- Performed initial data exploration using `df.head()` and `df.info()` to understand data structure, types, and identify potential issues.

### 2. Environment Setup
- Ensured all necessary libraries, including `cmdstanpy` (Prophet's backend) and `prophet`, were correctly installed and imported.

### 3. Global Data Preprocessing
- Aggregated the raw data by 'Date' to compute global sums for 'Confirmed', 'Deaths', 'Recovered', and 'Active' cases, creating `df_agg` for simplified global trend analysis.

### 4. Interactive Global Trend Visualization
- Utilized Plotly to generate an interactive line chart showcasing the global progression of 'Confirmed', 'Deaths', and 'Recovered' cases over time, offering a dynamic view of the pandemic's trajectory.

### 5. Time Series Forecasting with Prophet
- **Data Preparation**: Transformed aggregated data into the 'ds' (datestamp) and 'y' (target variable) format required by Prophet for 'Confirmed', 'Deaths', and 'Recovered' cases.
- **Model Training**: Initialized and trained separate Prophet models for each of the three key metrics.
- **Future Prediction**: Generated future dataframes extending 7 days beyond the last observed date.
- **Forecast Generation**: Produced 7-day forecasts, including point estimates (`yhat`) and uncertainty intervals (`yhat_lower`, `yhat_upper`).
- **Forecast Visualization**: Rendered interactive Plotly charts for each forecast, displaying historical data, future predictions, and confidence intervals.

## Results & Business Value
- **Clear Trend Identification**: Visualizations clearly illustrate the exponential growth of confirmed cases and deaths, alongside the encouraging increase in recoveries.
- **Short-term Predictive Insights**: The Prophet models provide reliable 7-day forecasts, offering actionable insights for healthcare systems, policymakers, and logistical planners to anticipate future demands and allocate resources effectively.
- **Interactive Data Exploration**: Plotly visualizations empower stakeholders to interact with the data, zoom into specific periods, and gain a deeper understanding of trends and predictions.

## Conclusion & Future Work
This project successfully leverages modern data science tools to analyze and forecast COVID-19 trends, delivering valuable insights for strategic planning. The methodology is robust and scalable, adaptable to various time-series prediction challenges.

**Future Enhancements could include:**
- **Feature Engineering**: Incorporating external factors like government policy changes, vaccination rates, and mobility data to enhance model accuracy.
- **Localized Forecasting**: Extending the analysis to specific countries or regions for more granular insights.
- **Model Evaluation & Optimization**: Implementing advanced validation techniques (e.g., cross-validation, backtesting) and hyperparameter tuning for Prophet models.
- **Dashboard Development**: Creating an interactive dashboard for real-time monitoring and scenario analysis.

## How to Run This Notebook
To replicate this analysis:
1.  **Clone the Repository**: Download the project files from GitHub.
2.  **Open in Google Colab**: Upload the `.ipynb` file to Google Colab or open it directly via a GitHub link.
3.  **Install Dependencies**: Run the cell containing `!pip install cmdstanpy prophet` to ensure all required libraries are installed.
4.  **Data Placement**: Ensure the `covid_19_clean_complete.csv` file is placed in the `/content/` directory of your Colab environment (or adjust the path in the data loading cell).
5.  **Execute Cells**: Run all cells sequentially to observe data loading, preprocessing, visualization, and forecasting results.
```

## Calculate Forecast Error

In [22]:
# Merge forecast with actual values for Confirmed cases
forecast_confirmed_with_actual = pd.merge(forecast_confirmed, df_agg[['Date', 'Confirmed']], left_on='ds', right_on='Date', how='inner')
forecast_confirmed_with_actual.rename(columns={'Confirmed': 'y_actual'}, inplace=True)

# Calculate Mean Absolute Error (MAE) for Confirmed cases
mae_confirmed = np.mean(np.abs(forecast_confirmed_with_actual['yhat'] - forecast_confirmed_with_actual['y_actual']))
print(f"Mean Absolute Error (MAE) for Confirmed Cases: {mae_confirmed:.2f}")

# Merge forecast with actual values for Deaths cases
forecast_deaths_with_actual = pd.merge(forecast_deaths, df_agg[['Date', 'Deaths']], left_on='ds', right_on='Date', how='inner')
forecast_deaths_with_actual.rename(columns={'Deaths': 'y_actual'}, inplace=True)

# Calculate Mean Absolute Error (MAE) for Deaths cases
mae_deaths = np.mean(np.abs(forecast_deaths_with_actual['yhat'] - forecast_deaths_with_actual['y_actual']))
print(f"Mean Absolute Error (MAE) for Deaths Cases: {mae_deaths:.2f}")

# Merge forecast with actual values for Recovered cases
forecast_recovered_with_actual = pd.merge(forecast_recovered, df_agg[['Date', 'Recovered']], left_on='ds', right_on='Date', how='inner')
forecast_recovered_with_actual.rename(columns={'Recovered': 'y_actual'}, inplace=True)

# Calculate Mean Absolute Error (MAE) for Recovered cases
mae_recovered = np.mean(np.abs(forecast_recovered_with_actual['yhat'] - forecast_recovered_with_actual['y_actual']))
print(f"Mean Absolute Error (MAE) for Recovered Cases: {mae_recovered:.2f}")

Mean Absolute Error (MAE) for Confirmed Cases: 42758.84
Mean Absolute Error (MAE) for Deaths Cases: 986.38
Mean Absolute Error (MAE) for Recovered Cases: 31832.41
